# NB_SEC_Catalog · P7 · catalogue harvest
Reads every table and column of each secured lakehouse (`sec.Item.OneLakeSecured = 1`) into `sec.TableCatalog` and `sec.ColumnCatalog`.
Nobody types table names. New tables arrive with `Reviewed = 0`, so no pattern rule reaches them until a steward reviews them (closed by default).
Column classes are only *suggested* here (from `sec.ClassRule`); a steward confirms them. Needs `P7_scale.sql` in SQLDB_SecMeta.

In [ ]:
ITEMS = ""   # optional: comma-separated lakehouse names; empty = every secured lakehouse

In [ ]:
%run NB_SEC_Common

In [ ]:
# 1. Which lakehouses, and their Fabric IDs (resolved by name, as in NB_SEC_Sync)
items = sql_df("SELECT i.ItemKey, i.ItemName, w.WorkspaceName FROM sec.Item i JOIN sec.Workspace w ON w.WorkspaceKey = i.WorkspaceKey WHERE i.OneLakeSecured = 1")
only = {x.strip() for x in ITEMS.split(",") if x.strip()}
if only: items = items[items.ItemName.isin(only)]
ws_id = {w["displayName"]: w["id"] for w in fabric_all("/workspaces")}
def find_item(ws, name):
    wid = ws_id.get(ws)
    if not wid: return None
    for x in fabric_all(f"/workspaces/{wid}/items"):
        if x["displayName"] == name: return wid, x["id"]
    return None

In [ ]:
# 2. List tables (schema-enabled: Tables/<schema>/<table>; schema-less: Tables/<table>) and read each Delta schema
def is_delta(path):
    try: return any(f.name.rstrip("/") == "_delta_log" for f in notebookutils.fs.ls(path))
    except Exception: return False

def harvest(wid, iid):
    base = f"abfss://{wid}@onelake.dfs.fabric.microsoft.com/{iid}/Tables"
    found = []
    for e in notebookutils.fs.ls(base):
        if not e.isDir: continue
        n = e.name.rstrip("/")
        if is_delta(e.path): found.append(("dbo", n, f"/Tables/{n}", e.path))
        else:
            for t in notebookutils.fs.ls(e.path):
                if t.isDir and is_delta(t.path):
                    tn = t.name.rstrip("/"); found.append((n, tn, f"/Tables/{n}/{tn}", t.path))
    tables, columns = [], []
    for schema, name, tpath, path in found:
        try:
            fields = spark.read.format("delta").load(path).schema.fields
        except Exception as ex:
            print("  could not read", tpath, str(ex)[:200]); continue
        tables.append({"TablePath": tpath, "SchemaName": schema, "TableName": name})
        for k, f in enumerate(fields, start=1):
            columns.append({"TablePath": tpath, "ColumnName": f.name, "DataType": f.dataType.simpleString()[:100], "Ordinal": k})
    return tables, columns

In [ ]:
# 3. Merge into the register in one statement per lakehouse (scales to hundreds of tables)
MERGE_TABLES = """
DECLARE @k INT = ?; DECLARE @j NVARCHAR(MAX) = ?;
MERGE sec.TableCatalog AS t
USING (SELECT TablePath, SchemaName, TableName FROM OPENJSON(@j) WITH (TablePath NVARCHAR(200), SchemaName NVARCHAR(100), TableName NVARCHAR(200))) AS s
ON t.ItemKey = @k AND t.TablePath = s.TablePath
WHEN MATCHED THEN UPDATE SET LastSeenAt = SYSUTCDATETIME(), IsPresent = 1
WHEN NOT MATCHED BY TARGET THEN INSERT (ItemKey, TablePath, SchemaName, TableName, Reviewed, ReviewedBy, ReviewedAt)
     VALUES (@k, s.TablePath, s.SchemaName, s.TableName,
             -- tables already covered by an explicit rule were reviewed when that rule was written
             CASE WHEN EXISTS (SELECT 1 FROM sec.ObjectRule o WHERE o.ItemKey = @k AND o.TablePath = s.TablePath) THEN 1 ELSE 0 END,
             CASE WHEN EXISTS (SELECT 1 FROM sec.ObjectRule o WHERE o.ItemKey = @k AND o.TablePath = s.TablePath) THEN 'explicit rule' END,
             CASE WHEN EXISTS (SELECT 1 FROM sec.ObjectRule o WHERE o.ItemKey = @k AND o.TablePath = s.TablePath) THEN SYSUTCDATETIME() END)
WHEN NOT MATCHED BY SOURCE AND t.ItemKey = @k THEN UPDATE SET IsPresent = 0;"""
MERGE_COLUMNS = """
DECLARE @k INT = ?; DECLARE @j NVARCHAR(MAX) = ?;
MERGE sec.ColumnCatalog AS t
USING (SELECT TablePath, ColumnName, DataType, Ordinal FROM OPENJSON(@j) WITH (TablePath NVARCHAR(200), ColumnName NVARCHAR(128), DataType NVARCHAR(100), Ordinal INT)) AS s
ON t.ItemKey = @k AND t.TablePath = s.TablePath AND t.ColumnName = s.ColumnName
WHEN MATCHED THEN UPDATE SET DataType = s.DataType, Ordinal = s.Ordinal, IsPresent = 1
WHEN NOT MATCHED BY TARGET THEN INSERT (ItemKey, TablePath, ColumnName, DataType, Ordinal) VALUES (@k, s.TablePath, s.ColumnName, s.DataType, s.Ordinal)
WHEN NOT MATCHED BY SOURCE AND t.ItemKey = @k THEN UPDATE SET IsPresent = 0;"""
SUGGEST = """
UPDATE c SET SuggestedClass = (SELECT TOP 1 r.ColumnClass FROM sec.ClassRule r WHERE LOWER(c.ColumnName) LIKE r.NamePattern ORDER BY r.Priority, r.NamePattern)
FROM sec.ColumnCatalog c WHERE c.ColumnClass IS NULL;"""

before = int(sql_exec("SELECT COUNT(*) FROM sec.TableCatalog", fetch_scalar=True))
for it in items.itertuples():
    ids = find_item(it.WorkspaceName, it.ItemName)
    if not ids: print("not found:", it.ItemName); continue
    tables, columns = harvest(*ids)
    sql_exec(MERGE_TABLES, (int(it.ItemKey), json.dumps(tables)))
    sql_exec(MERGE_COLUMNS, (int(it.ItemKey), json.dumps(columns)))
    audit("CATALOG_HARVEST", it.ItemName, after={"tables": len(tables), "columns": len(columns)})
    print(f"{it.ItemName}: {len(tables)} tables, {len(columns)} columns")
sql_exec(SUGGEST)
after = int(sql_exec("SELECT COUNT(*) FROM sec.TableCatalog", fetch_scalar=True))
print(f"Catalogue: {after} tables ({after - before} new)")

In [ ]:
# 4. What needs a steward now (new tables are closed until reviewed)
display(sql_df("SELECT * FROM sec.vw_ClassificationQueue ORDER BY FirstSeenAt DESC, TablePath, ColumnName"))